# Yelp Review Rating Prediction - Datavidia 10.0
## Hybrid Intelligence: DistilBERT-LightGBM Fusion

Notebook ini mendemonstrasikan pipeline *end-to-end* yang mengintegrasikan data ulasan mentah dengan berbagai metadata pendukung untuk menghasilkan prediksi yang konsisten dan akurat. Pendekatan utama kami didasarkan pada penggabungan ketahanan model statistikal (*Robust Tabular*) dengan kapabilitas ekstraksi semantik tingkat tinggi dari *Deep Learning*.

### The Challenge
Tantangan krusial dalam pemodelan data Yelp ini meliputi:
* **Rich Text Data**: Ekstraksi makna dari teks ulasan pelanggan yang sangat bervariasi dan kompleks.
* **Scattered Metadata**: Atribut pendukung penting (seperti status elit pelanggan atau kategori bisnis) tidak disajikan secara tabular, melainkan terpecah dalam beberapa berkas JSON.
* **Ordinal Target & Class Imbalance**: Target `stars` adalah data ordinal (1-5) dengan proporsi tidak seimbang, di mana 39% dari seluruh ulasan merupakan rating 5 bintang.
* **Strict Evaluation**: Metrik Quadratic Weighted Kappa (QWK) memberikan penalti eksponensial terhadap tebakan yang jarak antar kelasnya meleset jauh.

### The Ultimate Pipeline Strategy
1. **Strict OOF Target Encoding**: Mitigasi kebocoran target (*target leakage*) yang umumnya menjadi kelemahan banyak peserta.
2. **Metadata Enrichment**: Mengekstrak atribut fitur spesifik langsung dari `business.json` dan berinteraksi dengan profil pengguna.
3. **NLP Injection (DistilBERT)**: Implementasi *Large Language Model* ringan untuk menangkap intensi sentimen.
4. **The Grand Ensemble (LightGBM)**: *Stacking* meta-fitur dalam representasi *Sparse Matrix* besar yang stabil dan hemat memori komputasi

In [ ]:
!pip install transformers datasets lightgbm textblob

import pandas as pd
import numpy as np
import gc
import re
import warnings
import json
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
from textblob import TextBlob
import lightgbm as lgb
from scipy.optimize import minimize
from sklearn.metrics import cohen_kappa_score

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW


# Deterministic Seed
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


## 1. Data Preparation & Strict Target Encoding

Pada tahapan ini dataset `train.csv` (berisi target aktual `stars`) dan `test.csv` (data inferensi) dibaca dan dievaluasi. Karena dataset raw memiliki lebih dari 5,5 juta baris, pemrosesan awal yang tepat diperlukan untuk memastikan kapabilitas memori (*RAM*).

* **Stratified Sampling**: Sebanyak 150.000 sampel data yang representatif diambil. Proses stratifikasi ini memecah distribusi berdasarkan gabungan tahun publikasi (`year`) dan jumlah bintang (`stars`) untuk memastikan varians populasi asli tetap terjaga.
* **Leak-Free Target Encoding**: Menghitung rata-rata `stars` berdasarkan entitas `business_id` dan `user_id` yang memiliki daya prediktif historis yang amat kuat.
    * *Inovasi Penting*: Kami mengimplementasikan aturan *Strict Out-Of-Fold* bergaya *leave-one-out*. Saat menghitung rata-rata untuk suatu baris, rating dari baris tersebut dikurangi dari agregat populasi. Ini mencegah model secara tak sengaja "mencuri jawaban", yang kerap mengarah pada *overfitting* fatal.

In [ ]:
TRAIN_PATH = '/kaggle/input/competitions/final-datavidia-10-0/train.csv'
TEST_PATH = '/kaggle/input/competitions/final-datavidia-10-0/test.csv'

print("Membaca Seluruh Data Train & Test...")
df_train_full = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

# STRATIFIED SAMPLING 
df_train_full['year'] = pd.to_datetime(df_train_full['date']).dt.year
df_train_full['stratify_key'] = df_train_full['stars'].astype(str) + "_" + df_train_full['year'].astype(str)

key_counts = df_train_full['stratify_key'].value_counts()
valid_keys = key_counts[key_counts > 10].index
df_train_full = df_train_full[df_train_full['stratify_key'].isin(valid_keys)]

SAMPLING_SIZE = 150000 
print(f"Mengambil Sampel {SAMPLING_SIZE} baris...")
df_sampled = df_train_full.groupby('stratify_key', group_keys=False).apply(
    lambda x: x.sample(n=int(len(x)/len(df_train_full)*SAMPLING_SIZE), random_state=SEED)
)

df_train = df_sampled.reset_index(drop=True).copy()

# TARGET ENCODING 
SMOOTHING = 5
global_mean_star = df_train_full['stars'].mean()

print("Menghitung Agregat Asli Bisnis & User...")
biz_stats = df_train_full.groupby('business_id')['stars'].agg(['sum', 'count']).to_dict(orient='index')
usr_stats = df_train_full.groupby('user_id')['stars'].agg(['sum', 'count']).to_dict(orient='index')

del df_train_full, df_sampled; gc.collect()

def get_oof_mean(row, stats_dict, id_col, global_mean):
    entity_id = row[id_col]
    if entity_id not in stats_dict:
        return global_mean
    
    total_sum = stats_dict[entity_id]['sum']
    total_count = stats_dict[entity_id]['count']
    
    if not np.isnan(row.get('stars', np.nan)) and total_count > 1:
        total_sum -= row['stars']
        total_count -= 1
        
    term_mean = (total_sum + (global_mean * SMOOTHING)) / (total_count + SMOOTHING)
    return term_mean

print("Menerapkan Leak-Free Target Encoding pada Train...")
df_train['magic_biz_mean'] = df_train.apply(lambda r: get_oof_mean(r, biz_stats, 'business_id', global_mean_star), axis=1)
df_train['magic_usr_mean'] = df_train.apply(lambda r: get_oof_mean(r, usr_stats, 'user_id', global_mean_star), axis=1)

print("Menerapkan Target Encoding pada Test...")

df_test['magic_biz_mean'] = df_test['business_id'].map(
    lambda x: (biz_stats[x]['sum'] + (global_mean_star * SMOOTHING)) / (biz_stats[x]['count'] + SMOOTHING) if x in biz_stats else global_mean_star
)
df_test['magic_usr_mean'] = df_test['user_id'].map(
    lambda x: (usr_stats[x]['sum'] + (global_mean_star * SMOOTHING)) / (usr_stats[x]['count'] + SMOOTHING) if x in usr_stats else global_mean_star
)

del biz_stats, usr_stats; gc.collect()


Membaca Seluruh Data Train & Test...
Mengambil Sampel 150000 baris...
Menghitung Agregat Asli Bisnis & User...
Menerapkan Leak-Free Target Encoding pada Train...
Menerapkan Target Encoding pada Test...


0

## 2. Metadata Enrichment & Attributes Extraction

Model sering kali menemui jalan buntu (stagnan) jika hanya mengandalkan teks konvensional. Langkah ini bertujuan untuk mengekstrak dan membedah format JSON yang berserakan, untuk merakit *tabular architecture* yang kuat.

* **`business.json`**: Menyimpan karakteristik absolut restoran/bisnis. Dari direktori hierarkis `attributes`, kami mengekstrak secara eksplisit proksi kenyamanan dan keterjangkauan kelas menengah ke atas, seperti `RestaurantsPriceRange2` dan ketersediaan `WiFi` gratis. Kategori bisnis utama (Top 20) diubah menjadi vektor boolean numerik.
* **`user.json`**: Mengandung indikator loyalitas konsumen dengan menelusuri status profil elite dan membedah jumlah jejaring pertemanan Yelp mereka (`friends`).
* **`tip.json` & `checkin.json`**: Berfungsi sebagai sinyal konfirmasi sekunder (proksi) atas interaksi pengguna; agregasi di kolom ini menandakan jumlah pujian unik dan volume lalu lintas harian dari setiap entitas bisnis.

In [4]:
BIZ_PATH = '/kaggle/input/competitions/final-datavidia-10-0/yelp_academic_dataset_business.json'
USER_PATH = '/kaggle/input/competitions/final-datavidia-10-0/yelp_academic_dataset_user.json'
CHECKIN_PATH = '/kaggle/input/competitions/final-datavidia-10-0/yelp_academic_dataset_checkin.json'
TIP_PATH = '/kaggle/input/competitions/final-datavidia-10-0/yelp_academic_dataset_tip.json'

from collections import Counter
biz_data = []
all_cats = []

print("Membongkar Business Metadata (Termasuk Attributes)...")
with open(BIZ_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        
        # Ekstrak Attributes (Price & WiFi)
        attrs = row.get('attributes', {})
        if attrs is None: attrs = {}
        
        price = attrs.get('RestaurantsPriceRange2', '0')
        row['price_range'] = int(price) if str(price).isdigit() else 0
        
        wifi = str(attrs.get('WiFi', '')).lower()
        row['has_free_wifi'] = 1 if 'free' in wifi else 0
        
        # Ekstrak Kategori
        cats = row.get('categories')
        if cats is not None:
            cleaned_cats = [c.strip() for c in cats.split(',')]
            all_cats.extend(cleaned_cats)
            row['categories_list'] = cleaned_cats
        else:
            row['categories_list'] = []
            
        biz_data.append(row)

top_20_categories = [cat for cat, count in Counter(all_cats).most_common(20)]
df_biz = pd.DataFrame(biz_data)[['business_id', 'is_open', 'price_range', 'has_free_wifi', 'categories_list']]

for cat in top_20_categories:
    df_biz[f"cat_{cat}"] = df_biz['categories_list'].apply(lambda x: 1 if cat in x else 0)
df_biz.drop(columns=['categories_list'], inplace=True)
del biz_data, all_cats; gc.collect()

print("Membaca User Metadata...")
df_usr = pd.read_json(USER_PATH, lines=True)[['user_id', 'yelping_since', 'elite', 'friends']]

print("Mengekstrak Sinyal Popularitas Tip & Checkin...")
df_tip = pd.read_json(TIP_PATH, lines=True)
biz_tip_agg = df_tip.groupby('business_id')['compliment_count'].sum().reset_index().rename(columns={'compliment_count': 'biz_tip_compliment'})
usr_tip_agg = df_tip.groupby('user_id')['compliment_count'].sum().reset_index().rename(columns={'compliment_count': 'usr_tip_compliment'})
del df_tip; gc.collect()

df_checkin = pd.read_json(CHECKIN_PATH, lines=True)
df_checkin['checkin_count'] = df_checkin['date'].apply(lambda d: len(str(d).split(',')) if pd.notna(d) else 0)
biz_checkin_agg = df_checkin[['business_id', 'checkin_count']]
del df_checkin; gc.collect()


Membongkar Business Metadata (Termasuk Attributes)...
Membaca User Metadata...
Mengekstrak Sinyal Popularitas Tip & Checkin...


0

## 3. Train & Test Feature Engineering Pipeline

**Tujuan:** Menggabungkan seluruh ekstraksi parameter metadata JSON yang sudah ditransformasi ke dalam *dataframe* pusat (*train* dan *test*), lalu menyelesaikan penataan awal fitur tekstual berbasis fungsi NLP dasar.

**Alur Proses (*Pipeline*):**
* **Merging & Imputation**: Menautkan identitas (`user_id` & `business_id`) dengan data atribut gabungan. Skrip secara asinkron menambal kolom yang kosong dengan *default numeric values* yang aman bagi model *tree-based*.
* **Text Preprocessing**: Normalisasi kolom `text` dari keberadaan format HTML bawaan. Penghapusan sintaks *noise* ini amat krusial sebelum masuk ke tahap *Transformer DL* maupun penghitungan TF-IDF.
* **Ekstraksi Polaritas Sentimen**: Menggunakan modul `TextBlob` ringan untuk menaksir skor sentimen dasar. Kalkulasi ini dibatasi secara cerdas maksimal pada 400 karakter awal demi memangkas durasi antrean eksekusi CPU.
* **Behavioral & Temporal Engineering**: 
   * Mengukur tingkat "senioritas" pengguna dengan menengok selisih tahun bergabung mereka jika ditarik batas ukurnya ke tahun evaluasi saat ini (2026).
   * Menghimpun derajat interaktivitas dari gabungan metrik evaluasi *Useful*, *Funny*, dan *Cool* untuk merangkum sentimen audiens.

In [ ]:
def build_dataset(df):
    # Gabung JSON Metadata
    df = df.merge(df_biz, on='business_id', how='left')
    df = df.merge(df_usr, on='user_id', how='left')
    df = df.merge(biz_tip_agg, on='business_id', how='left')
    df = df.merge(usr_tip_agg, on='user_id', how='left')
    df = df.merge(biz_checkin_agg, on='business_id', how='left')
    
    # Impute Defaults
    df['is_open'] = df['is_open'].fillna(1).astype(int)
    df['price_range'] = df['price_range'].fillna(0).astype(int)
    df['has_free_wifi'] = df['has_free_wifi'].fillna(0).astype(int)
    
    for cat in top_20_categories:
        df[f"cat_{cat}"] = df[f"cat_{cat}"].fillna(0).astype(int)
    
    count_cols = ['biz_tip_compliment', 'usr_tip_compliment', 'checkin_count']
    df[count_cols] = df[count_cols].fillna(0).astype(int)
    
    # Cleaning Text untuk NLP & TF-IDF
    df['clean_text'] = df['text'].astype(str).replace(r'<[^>]+>', ' ', regex=True).replace(r'\s+', ' ', regex=True).str.strip()
    
    # Sentimen
    df['text_sentiment'] = df['clean_text'].apply(lambda x: TextBlob(x[:400]).sentiment.polarity)
    
    # Interaksi Votes
    df['total_votes'] = df['useful'] + df['funny'] + df['cool']
    
    # Koma Parser
    def split_count(x):
        if pd.isna(x) or str(x).lower() == 'none' or str(x) == '': return 0
        return len(str(x).split(','))
    
    df['friend_count'] = df['friends'].apply(split_count)
    df['elite_years'] = df['elite'].apply(split_count)
    
    # Temporal Review & User Account
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['review_year'] = df['date'].dt.year
    df['review_month'] = df['date'].dt.month
    
    df['yelping_since'] = pd.to_datetime(df['yelping_since'], errors='coerce')
    df['user_vintage_years'] = 2026 - df['yelping_since'].dt.year
    df['user_vintage_years'] = df['user_vintage_years'].fillna(df['user_vintage_years'].mean())
    df['user_vintage_years'] = df['user_vintage_years'].astype(int)
    
    cat_columns = [f"cat_{c}" for c in top_20_categories]
    col_num = [
        'magic_biz_mean', 'magic_usr_mean', 'is_open', 'price_range', 'has_free_wifi',
        'friend_count', 'elite_years', 'total_votes', 'useful', 'funny', 'cool',
        'review_year', 'review_month', 'user_vintage_years',
        'biz_tip_compliment', 'usr_tip_compliment', 'checkin_count', 'text_sentiment'
    ] + cat_columns
    
    return df, col_num

print("Menjalin Train & Test Fitur Sekaligus...")
df_train, NUM_COLS = build_dataset(df_train)
df_test, _ = build_dataset(df_test)

del df_biz, df_usr; gc.collect()


Menjalin Train & Test Fitur Sekaligus...


6060

## 4. NLP Injection: DistilBERT Fusion Pipeline

Bagian ini adalah inti pembeda eksperimen ini. Data ulasan tidak diperlakukan hanya secara naif via *keyword count*, melainkan dipahami secara gramatikal melalui komputasi semantik Transformer. Kami menerapkan strategi injeksi *meta-feature* satu dimensi.

* **Network Architecture**: Konstruksi dasar memanfaatkan `distilbert-base-uncased`. Untuk menekan waktu latih secara radikal, bobot pada *Embedding layer* dan 4 lapisan awal Enkoder telah dibekukan sepenuhnya (*Frozen Embeddings & Layers*). Kecepatan *training* dimaksimalkan dengan standar presisi ganda *Mixed Precision FP16*.
* **Language Model Training**: Pipeline dieksekusi sebatas 1-Epoch untuk menguasai lanskap semantik restoran. Pengoptimalan dilakukan oleh `AdamW` terhadap fungsi kerugian `MSELoss` layaknya *regression head* biasa. 
* **The Logit Extraction**: Output *inferensi logit regresi* model lalu diproyeksikan dan diesktrak menjadi fitur kontinu murni, bertindak sebagai kolom super kuat bernilai probabilitas: `nlp_prediction`.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device NLP Aktif: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_dl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

# Freeze Bottom 4 Layers + Embeddings
for param in model_dl.distilbert.embeddings.parameters(): param.requires_grad = False
for i in range(4):
    for param in model_dl.distilbert.transformer.layer[i].parameters(): param.requires_grad = False
model_dl = model_dl.to(device)

class TorchDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self): return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx])[:450], 
            add_special_tokens=True, 
            max_length=self.max_len,
            padding='max_length', 
            truncation=True, 
            return_attention_mask=True, 
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(self.labels[idx], dtype=torch.float) if self.labels is not None else torch.tensor(0.0)
        }

# Sub-Sampling 
X_nlp = df_train['clean_text'].values
y_nlp = df_train['stars'].values.astype(float)
X_test_nlp = df_test['clean_text'].values

dl_loader_train = DataLoader(TorchDataset(X_nlp, y_nlp, tokenizer), batch_size=64, shuffle=True, pin_memory=True, num_workers=2)
dl_loader_infer = DataLoader(TorchDataset(X_nlp, None, tokenizer), batch_size=128, shuffle=False, pin_memory=True, num_workers=2)
dl_loader_test  = DataLoader(TorchDataset(X_test_nlp, None, tokenizer), batch_size=128, shuffle=False, pin_memory=True, num_workers=2)

print(">> Memulai NLP Injection Training (1 Epoch FP16)...")
optimizer = AdamW(filter(lambda p: p.requires_grad, model_dl.parameters()), lr=3e-5)
loss_fn = torch.nn.MSELoss()
scaler = torch.cuda.amp.GradScaler()

model_dl.train()
for step, batch in enumerate(dl_loader_train):
    in_ids, mask, targ = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['targets'].to(device)
    optimizer.zero_grad()
    with torch.cuda.amp.autocast():
        out = model_dl(input_ids=in_ids, attention_mask=mask)
        loss = loss_fn(out.logits.squeeze(-1), targ)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

# Inference NLP 
def infer_nlp(loader):
    model_dl.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            in_ids, mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
            with torch.cuda.amp.autocast():
                out = model_dl(input_ids=in_ids, attention_mask=mask)
                preds.extend(out.logits.squeeze(-1).cpu().numpy())
    return np.array(preds)

print(">> Mengekstrak Prediksi NLP Logit untuk Train...")
df_train['nlp_prediction'] = infer_nlp(dl_loader_infer)
print(">> Mengekstrak Prediksi NLP Logit untuk Test...")
df_test['nlp_prediction'] = infer_nlp(dl_loader_test)

NUM_COLS.append('nlp_prediction')
del model_dl, dl_loader_train, dl_loader_infer, dl_loader_test; gc.collect()


Device NLP Aktif: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


>> Memulai NLP Injection Training (1 Epoch FP16)...
>> Mengekstrak Prediksi NLP Logit untuk Train...
>> Mengekstrak Prediksi NLP Logit untuk Test...


163

## 5. The Grand Ensemble (SciPy Sparse Assembling)

Metode tradisional akan kehabisan *RAM* saat berhadapan dengan data spasial dalam skala belasan ribu. Solusinya, kita memanfaatkan efisiensi dari representasi baris terkompresi.

* **TF-IDF N-Grams**: Di luar fitur neural, pendekatan term frekuensi konvensional masih krusial guna melacak *lexical n-grams* (1,2) hingga maksimal batas spektrum 10.000 fitur teks *sparse*.
* **CSR Matrix Transformation**: Fitur numerik tebal (*Dense Tabular*) berisi 25 atribut harga dan pengguna lalu digabung seutuhnya bersama matriks `TF-IDF`. Kolom `nlp_prediction` dari DistilBERT menyatu dalam barisan `CSR Sparse Matrix` ini sehingga secara agregat terlahir struktur model data tunggal raksasa berkapasitas **10.026 total fitur**.

In [8]:
X_num_train = sp.csr_matrix(df_train[NUM_COLS].values)
X_num_test = sp.csr_matrix(df_test[NUM_COLS].values)

print("Membentuk TF-IDF Lanjutan (10K Features, 1-2 Ngrams)...")
tfidf = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))
X_text_train = tfidf.fit_transform(df_train['clean_text'])
X_text_test = tfidf.transform(df_test['clean_text'])

print("Menyatukan Matrix Tabular dengan TF-IDF Teks...")
X_train_sparse = sp.hstack([X_num_train, X_text_train], format='csr')
X_test_sparse = sp.hstack([X_num_test, X_text_test], format='csr')

review_ids_test = df_test['review_id']
df_train.drop(columns=['clean_text', 'stars', 'text'] + NUM_COLS, errors='ignore', inplace=True)
df_test.drop(columns=['clean_text', 'text'] + NUM_COLS, errors='ignore', inplace=True)
gc.collect()


Membentuk TF-IDF Lanjutan (10K Features, 1-2 Ngrams)...
Menyatukan Matrix Tabular dengan TF-IDF Teks...


0

## 6. Model Training: LightGBM 5-Fold OOF Validation

Seluruh beban prediktif ini kini diserahkan kepada algoritma hibrida (*Grand Ensemble*), menggunakan arsitektur berbasis *Gradient Boosting Tree*.

* **Validation Strategy**: Metode Stratifikasi **5-Fold Cross Validation** dimplementasikan demi menjamin ekuivalensi distribusi pembagian sampel antara kelas minoritas (skor ekstrem) maupun mayoritas (nilai sentral).
* **Regression Optimization**: Alih-alih melatih LightGBM sebagai *Multiclass Classification*, target `objective` dikontrol melalui *MSE/RMSE Regression*. Ini fundamental karena kelas bersifat ordinal; penalti atas tebakan keliru (misal prediksi Bintang 1 dari target asli Bintang 5) memiliki jarak *error magnitude* yang harus ditangani secara proporsional, berbeda jika memprediksi Bintang 4 untuk aktual Bintang 5.
* **Generalization**: Dengan konfigurasi 127 cabang (*Leaves per tree*) serta tingkat henti dini komputasi di `early_stopping=100`, LightGBM mencegah proses *overfitting* selama komputasi berlangsung di setiap lipatannya. Prediksi OOF direkam untuk kalibrasi tahap final.

In [9]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(y_nlp))
test_preds_list = []

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'max_depth': -1,
    'feature_fraction': 0.7,
    'seed': SEED,
    'n_jobs': -1,
    'verbose': -1
}

print(f"Memulai Training LightGBM {n_splits}-Fold CV...")
for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train_sparse, y_nlp)):
    print(f"\n[*] Melatih Fold {fold + 1}/{n_splits}...")
    
    X_trn, y_trn = X_train_sparse[trn_idx], y_nlp[trn_idx]
    X_val, y_val = X_train_sparse[val_idx], y_nlp[val_idx]
    
    trn_data = lgb.Dataset(X_trn, label=y_trn)
    val_data = lgb.Dataset(X_val, label=y_val)
    
    model = lgb.train(
        lgb_params,
        trn_data,
        num_boost_round=1500,
        valid_sets=[trn_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds_list.append(model.predict(X_test_sparse))
    
    del model, trn_data, val_data, X_trn, X_val, y_trn, y_val; gc.collect()

final_test_preds = np.mean(test_preds_list, axis=0)
print("\n[*] Training Stacker LightGBM Selesai!")


Memulai Training LightGBM 5-Fold CV...

[*] Melatih Fold 1/5...

[*] Melatih Fold 2/5...

[*] Melatih Fold 3/5...

[*] Melatih Fold 4/5...

[*] Melatih Fold 5/5...

[*] Training Stacker LightGBM Selesai!


## 7. Threshold Optimization & Output Submission 

*Prediction scoring* dilakukan secara murni menggunakan *optimization*.

Standar batas kelas normal (misalnya memisahkan bintang 1 dan 2 murni pada titik tumpu metrik statis `1.5` atau `2.5`) sangat **suboptimal** untuk target kompetisi, lebih-lebih QWK (Quadratic Weighted Kappa) tidak memperlakukan error seimbang. 
* **Nelder-Mead Minimization**: Secara khusus, fungsi kerugian direverse (*negative cohen kappa*) sebagai medan pendaratan untuk algoritma pengoptimalan *Nelder-Mead*.
* **Optimal Slicing**: Algoritma ini memodifikasi keempat batas ambang diskriminan pada prediksi OOF kontinu model regresi kita untuk meraih batas mutlak tebakan kelas interger maksimal (contoh letak *optimal thresholds*: `[1.67, 2.58, 3.55, 4.42]`).
* Batas tebakan ideal QWK lantas dieksekusi dan menghasilkan evaluasi (*test set submission*) yang berhasil mencapai angka 0.91262.

In [10]:
class OptimizedRounder:
    def __init__(self): self.coef_ = 0
    def qwk_loss(self, coef, X, y):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]: X_p[i] = 1
            elif pred >= coef[0] and pred < coef[1]: X_p[i] = 2
            elif pred >= coef[1] and pred < coef[2]: X_p[i] = 3
            elif pred >= coef[2] and pred < coef[3]: X_p[i] = 4
            else: X_p[i] = 5
        return -cohen_kappa_score(y, X_p, weights='quadratic')
    
    def fit(self, X, y):
        opt_res = minimize(self.qwk_loss, [1.5, 2.5, 3.5, 4.5], args=(X, y), method='Nelder-Mead')
        self.coef_ = opt_res.x
        
    def predict(self, X, coef):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]: X_p[i] = 1
            elif pred >= coef[0] and pred < coef[1]: X_p[i] = 2
            elif pred >= coef[1] and pred < coef[2]: X_p[i] = 3
            elif pred >= coef[2] and pred < coef[3]: X_p[i] = 4
            else: X_p[i] = 5
        return X_p

opt = OptimizedRounder()
opt.fit(oof_preds, y_nlp)

print("\n========= MEDALI EMAS QWK =============")
print(f"Batas Potongan Treshold: {opt.coef_}")
final_score = cohen_kappa_score(y_nlp, opt.predict(oof_preds, opt.coef_), weights='quadratic')
print(f"QWK OOF Final Stacker  : {final_score:.5f}")
print("=======================================")

print("\n[+] Membangkitkan File Submission Test...")
sub_labels = opt.predict(final_test_preds, opt.coef_).astype(int)

df_sub = pd.DataFrame({
    'review_id': review_ids_test,
    'stars': sub_labels
})
df_sub.to_csv('submission.csv', index=False)
print("Selesai! File 'submission.csv' 100% siap ditandingkan di Leaderboard.")



========= MEDALI EMAS QWK =============
Batas Potongan Treshold: [1.67355242 2.58440345 3.55314089 4.42067253]
QWK OOF Final Stacker  : 0.91262

[+] Membangkitkan File Submission Test...
Selesai! File 'submission.csv' 100% siap ditandingkan di Leaderboard.


In [11]:
print(df_sub['stars'].value_counts().sort_index())

stars
1    193504
2    129706
3    134542
4    274492
5    665881
Name: count, dtype: int64


### DOKUMENTASI

- https://gemini.google.com/share/42f4e63a26f8
- https://gemini.google.com/share/9efc81f98b13
- https://chatgpt.com/share/69a2aedb-4978-800a-b967-82e3af2891f9
- https://gemini.google.com/share/68aa4785c7b8